# Re-run video processing

We re-run processing over all files listed in the files_with_proboscis.csv.

We use the _process_video function from choice_assay_pose_processor.py to run two ML models over each video:
- firstly we grab the first frame as an image and run an ML model to find the location of the feeding tubes
- secondly we run an ML model over every frame to identify the location of any bee present (max of 1)


In [1]:
import platform
import shutil
from concurrent.futures import ThreadPoolExecutor, TimeoutError
from dataclasses import replace
from datetime import UTC, datetime
from pathlib import Path
from time import perf_counter

import pandas as pd

from choice_assay.rpi.choice_assay_pose_processor import (
    DEFAULT_CHOICE_ASSAY_POSE_PROCESSOR_CFG,
    ChoiceAssayPoseProcessor,
    ChoiceAssayPoseProcessorCfg,
)

Logging expidite to default file: C:\Users\bee-ops\AppData\Local\Temp\expidite\20260716T134942283\logs\default_20260716T134942295.log at level 20
2026-07-16 15:49:42,297 expidite INFO   [24508] Loading C:\Users\bee-ops\.expidite\system.cfg...
Logging choice_assay to default file: C:\Users\bee-ops\AppData\Local\Temp\expidite\20260716T134942283\logs\default_20260716T134942295.log at level 20


In [2]:
# Required config
AZURE_KEYS_FILE = Path.home() / ".expidite" / "keys_choiceassay.env"

CONTAINER_NAME = "expidite-choiceassay-trapcam"
TYPE_ID = "CAVIDEO"

# Decide if we're running on Windows or Linux, and set the NAS root path accordingly
if "Linux" in platform.platform():
    running_on_linux = True
else:
    running_on_linux = False

# Local download directory (relative to notebook working directory)
if running_on_linux:
    nas_root = Path("/bee-ops-disk/")
    tmp_root = Path("/tmp/")
else:
    nas_root = Path("B://")
    tmp_root = Path.home() / "AppData" / "Local" / "Temp"

    # If we don't have access to the B: drive fall back to a local alternative
    if not nas_root.exists():
        print(f"Warning: B: drive not found, falling back to local alternative")
        nas_root = Path.home() / "bee-ops-disk"

DOWNLOAD_DIR = nas_root / "azure" / "choice_assay" / "expidite-choiceassay-trapcam"
OUTPUT_DIR = nas_root / "results" / "choice_assay_rerun"

DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Fast local cache for JIT staging before ML processing
LOCAL_CACHE_DIR = tmp_root / "choice_assay_video_cache"
LOCAL_CACHE_DIR.mkdir(parents=True, exist_ok=True)

# Number of files to keep prefetched ahead of the current inference file
PREFETCH_AHEAD = 6

# Diagnostics for silent waits while staging files from NAS to local cache
PREFETCH_WAIT_LOG_INTERVAL_SECONDS = 30
PREFETCH_HARD_TIMEOUT_SECONDS = None  # e.g. 900 to fail after 15 min

PREFIX = f"V3_{TYPE_ID}_"
SUFFIX = ".mp4"

OUTPUT_PREFIX_LEN = len("V3_CAVIDEO_d83add1a11c5_00_00_20260317")

print(f"Downloading {PREFIX} files from '{CONTAINER_NAME}' to {DOWNLOAD_DIR.resolve()}")
print(f"JIT local cache directory: {LOCAL_CACHE_DIR.resolve()}")

JIT local cache directory: C:\Users\bee-ops\AppData\Local\Temp\choice_assay_video_cache


In [3]:
# Create a CSV file of all the files in the container.  About 3m for 80k files.
"""
files = DOWNLOAD_DIR.glob(f"{PREFIX}*{SUFFIX}")

# Save the files list to file
files_df = pd.DataFrame([str(f) for f in files], columns=["filename"])
files_df.to_csv("files_list.csv", index=False)
"""

'\nfiles = DOWNLOAD_DIR.glob(f"{PREFIX}*{SUFFIX}")\n\n# Save the files list to file\nfiles_df = pd.DataFrame([str(f) for f in files], columns=["filename"])\nfiles_df.to_csv("files_list.csv", index=False)\n'

In [4]:
PROCESSED_LOG_PATH = OUTPUT_DIR / "processed_videos_log.csv"


def list_processed_videos() -> set[str]:
    """List videos already handled in prior runs, including no-detection outcomes."""
    if not PROCESSED_LOG_PATH.exists():
        return set()

    df = pd.read_csv(PROCESSED_LOG_PATH)
    required_cols = {"video_filename", "status"}
    if not required_cols.issubset(df.columns):
        print(
            f"Warning: {PROCESSED_LOG_PATH} missing required columns {required_cols}; ignoring log."
        )
        return set()

    terminal_statuses = {
        "processed_with_detection",
        "processed_no_detection",
        "missing_prefetch",
        "missing_processing",
    }
    done = df.loc[df["status"].isin(terminal_statuses), "video_filename"].dropna().astype(str)
    return set(done.tolist())


def record_video_attempt(
    video_fname: str,
    status: str,
    rows_saved: int = 0,
    note: str = "",
):
    """Append a processing outcome record so reruns can skip already handled videos."""
    record = pd.DataFrame(
        [
            {
                "video_filename": video_fname,
                "status": status,
                "rows_saved": rows_saved,
                "processed_at_utc": datetime.now(UTC).isoformat(),
                "note": note,
            }
        ]
    )
    record.to_csv(
        PROCESSED_LOG_PATH,
        mode="a",
        header=not PROCESSED_LOG_PATH.exists(),
        index=False,
    )


def save_results_to_csv(results: pd.DataFrame, video_fname: str) -> int:
    """Save the results to a CSV file and return the number of rows written."""
    if results.empty:
        print("No results to save.")
        return 0

    output_csv = OUTPUT_DIR / f"{video_fname[:OUTPUT_PREFIX_LEN]}.csv"
    if not output_csv.exists():
        # Create the output directory if it doesn't exist
        output_csv.parent.mkdir(parents=True, exist_ok=True)

    # Save to CSV: append if exists, otherwise create new
    if output_csv.exists():
        results.to_csv(output_csv, mode="a", header=False, index=False)
    else:
        results.to_csv(output_csv, index=False)

    return len(results)


def create_cfg() -> ChoiceAssayPoseProcessorCfg:
    """Create a configuration object for the choice assay pose processor."""
    # we override the sample_probability for the marked up video stream to 0 to avoid generating unnecessary videos during processing
    cfg = DEFAULT_CHOICE_ASSAY_POSE_PROCESSOR_CFG
    marked_up_output = cfg.outputs[1]
    marked_up_output.sample_probability = 0
    return replace(cfg, outputs=[cfg.outputs[0], marked_up_output])


def stage_video_to_local(video_path: Path) -> tuple[Path, float, int, int]:
    """Copy video from NAS to local cache and return path, copy_seconds, remote_size, local_size."""
    if not video_path.exists():
        raise FileNotFoundError(f"Remote video not found: {video_path}")

    local_path = LOCAL_CACHE_DIR / video_path.name
    remote_size = video_path.stat().st_size
    t_copy_start = perf_counter()
    if not local_path.exists():
        shutil.copy2(video_path, local_path)
    copy_seconds = perf_counter() - t_copy_start
    local_size = local_path.stat().st_size
    return local_path, copy_seconds, remote_size, local_size

In [6]:
# Run ML processing over all the video files in files_list.csv using JIT local staging.
# We prefetch a few files from NAS to local disk so inference reads local files.

# Read in the files to process
# The files_with_proboscis.csv is just a list of filenames, rather than a CSV
# Enable re-run by loading processing log and filtering already handled files.
processed_videos = set(list_processed_videos())
raw_file_list = set(pd.read_csv("files_with_proboscis.csv", header=None)[0].to_list())
videos_to_process = sorted(raw_file_list - processed_videos)

# Apply the DOWNLOAD_DIR path to the filenames to get full paths
videos_to_process = [DOWNLOAD_DIR / fname for fname in videos_to_process]

print(f"Found {len(videos_to_process)} new videos to process. {len(processed_videos)} already handled.")
print(f"Prefetch ahead: {PREFETCH_AHEAD} files")
print(f"Processing log: {PROCESSED_LOG_PATH.resolve()}")

processor = ChoiceAssayPoseProcessor(create_cfg(), 0)
start_time = datetime.now(UTC)


def _submit_prefetch(
    executor: ThreadPoolExecutor,
    queue: dict[int, object],
    submitted_at: dict[int, float],
    index: int,
):
    if 0 <= index < len(videos_to_process) and index not in queue:
        remote_path = Path(videos_to_process[index])
        queue[index] = executor.submit(stage_video_to_local, remote_path)
        submitted_at[index] = perf_counter()


def _prefetch_queue_snapshot(
    queue: dict[int, object],
    submitted_at: dict[int, float],
    limit: int = 6,
) -> str:
    pending = []
    now = perf_counter()
    for idx in sorted(queue.keys())[:limit]:
        fut = queue[idx]
        age = now - submitted_at.get(idx, now)
        if fut.done():
            state = "done"
        elif fut.running():
            state = "running"
        else:
            state = "pending"
        pending.append(f"{idx}:{state}:{age:.1f}s")
    suffix = " ..." if len(queue) > limit else ""
    return ", ".join(pending) + suffix


prefetch_futures: dict[int, object] = {}
prefetch_submitted_at: dict[int, float] = {}
copy_seconds_total = 0.0
prefetch_wait_seconds_total = 0.0
inference_seconds_total = 0.0
skipped_missing_prefetch = 0
skipped_missing_processing = 0
processed_success = 0

with ThreadPoolExecutor(max_workers=2) as pool:
    # Prime the local staging queue.
    for idx in range(min(PREFETCH_AHEAD, len(videos_to_process))):
        _submit_prefetch(pool, prefetch_futures, prefetch_submitted_at, idx)

    for i in range(len(videos_to_process)):
        _submit_prefetch(pool, prefetch_futures, prefetch_submitted_at, i + PREFETCH_AHEAD)

        remote_video = Path(videos_to_process[i])
        future = prefetch_futures[i]
        wait_start = perf_counter()
        local_video = None

        while True:
            try:
                local_video, copy_seconds, remote_size, local_size = future.result(
                    timeout=PREFETCH_WAIT_LOG_INTERVAL_SECONDS
                )
                prefetch_wait_seconds_total += perf_counter() - wait_start
                break
            except TimeoutError:
                wait_so_far = perf_counter() - wait_start
                snapshot = _prefetch_queue_snapshot(prefetch_futures, prefetch_submitted_at)
                print(
                    f"Waiting on prefetch index={i} file={remote_video.name} for {wait_so_far:.1f}s"
                    f" | queue={len(prefetch_futures)} [{snapshot}]"
                )
                if PREFETCH_HARD_TIMEOUT_SECONDS is not None and wait_so_far > PREFETCH_HARD_TIMEOUT_SECONDS:
                    raise TimeoutError(
                        f"Prefetch wait exceeded {PREFETCH_HARD_TIMEOUT_SECONDS}s for {remote_video}"
                    )
            except FileNotFoundError as err:
                prefetch_wait_seconds_total += perf_counter() - wait_start
                skipped_missing_prefetch += 1
                record_video_attempt(
                    remote_video.name,
                    status="missing_prefetch",
                    rows_saved=0,
                    note=str(err),
                )
                print(
                    f"Skipping missing file during prefetch ({skipped_missing_prefetch}): {remote_video.name}"
                    f" | reason={err}"
                )
                break

        prefetch_futures.pop(i, None)
        prefetch_submitted_at.pop(i, None)

        if local_video is None:
            continue

        copy_seconds_total += copy_seconds

        if not local_video.exists():
            skipped_missing_processing += 1
            msg = f"Local cache file missing before processing: {local_video}"
            record_video_attempt(
                remote_video.name,
                status="missing_processing",
                rows_saved=0,
                note=msg,
            )
            print(
                f"Skipping missing local cache file during processing ({skipped_missing_processing}): {local_video}"
            )
            continue

        # Process local file; keep original filename for output naming.
        t_infer_start = perf_counter()
        try:
            df = processor._process_video_file(local_video)
            inference_seconds = perf_counter() - t_infer_start
            inference_seconds_total += inference_seconds
            rows_saved = save_results_to_csv(df, remote_video.name)
            status = "processed_with_detection" if rows_saved > 0 else "processed_no_detection"
            record_video_attempt(remote_video.name, status=status, rows_saved=rows_saved)
            processed_success += 1
        except FileNotFoundError as err:
            skipped_missing_processing += 1
            record_video_attempt(
                remote_video.name,
                status="missing_processing",
                rows_saved=0,
                note=str(err),
            )
            print(
                f"Skipping file missing during processing ({skipped_missing_processing}): {remote_video.name}"
                f" | reason={err}"
            )
            continue
        finally:
            # Keep cache size bounded by removing files right after processing.
            try:
                local_video.unlink(missing_ok=True)
            except OSError as err:
                print(f"Warning: failed to remove cache file {local_video}: {err}")

        elapsed = (datetime.now(UTC) - start_time).total_seconds()
        n = i + 1
        print(
            f"{n}/{len(videos_to_process)} @ {elapsed / n:.1f} secs/video"
            f" | infer={inference_seconds:.2f}s"
            f" | copy={copy_seconds:.2f}s ({remote_size / 1e6:.1f}MB)"
            f" | avg_copy={copy_seconds_total / n:.2f}s"
            f" | avg_wait={prefetch_wait_seconds_total / n:.2f}s"
            f" | avg_infer={inference_seconds_total / n:.2f}s"
            f" | ok={processed_success}"
            f" | miss_prefetch={skipped_missing_prefetch}"
            f" | miss_process={skipped_missing_processing}"
            f" | cache_queue={len(prefetch_futures)}"
            f" | processed {remote_video.name} | output -> {OUTPUT_DIR.resolve()}"
        )

print(
    f"Completed rerun: ok={processed_success}, "
    f"missing_in_prefetch={skipped_missing_prefetch}, "
    f"missing_in_processing={skipped_missing_processing}"
)

Found 13917 new videos to process. 13 already handled.
Prefetch ahead: 6 files
Processing log: C:\Users\bee-ops\bee-ops-disk\results\choice_assay_rerun\processed_videos_log.csv
2026-07-16 15:52:59,954 choice_assay INFO   [24508] Pose model diagnostics: ultralytics=8.4.21 model_path=C:\Users\bee-ops\code\ChoiceAssay\src\choice_assay\resources\bee_best.pt names_count=1 names_keys=[0]
2026-07-16 15:53:33,647 choice_assay INFO   [24508] Pose timings: video=C:\Users\bee-ops\AppData\Local\Temp\choice_assay_video_cache\V3_CAVIDEO_d83add1a11c5_00_00_20260617T142604277_20260617T142842874.mp4 model_load=0.118s tube_detect=0.156s predict_setup=0.039s stream_iter=33.493s dataframe=0.003s markup_save=0.000s total=33.810s frames=794 rows=245 rows_per_frame=0.309
1/13917 @ 33.8 secs/video | infer=33.81s | copy=0.00s (3.5MB) | avg_copy=0.00s | avg_wait=0.00s | avg_infer=33.81s | ok=1 | miss_prefetch=0 | miss_process=0 | cache_queue=6 | processed V3_CAVIDEO_d83add1a11c5_00_00_20260617T142604277_2026061

KeyboardInterrupt: 

In [ ]:
csv_paths = sorted(OUTPUT_DIR.rglob("*.csv"))
print(f"CSV files available locally: {len(csv_paths)}")

df_list = []
for csv_path in csv_paths:
    df = pd.read_csv(csv_path)
    if not df.empty:
        df["source_file"] = csv_path.name
        df_list.append(df)

if df_list:
    aggregated_df = pd.concat(df_list, ignore_index=True)
else:
    aggregated_df = pd.DataFrame()

print(f"Aggregated rows: {len(aggregated_df)}")
aggregated_df.head()

In [ ]:
# Data validation before behaviour classification
required_columns = ["Tube_prob_likelihood", "End_prob_likelihood"]
missing_columns = [col for col in required_columns if col not in aggregated_df.columns]

if aggregated_df.empty:
    print("Validation: aggregated_df is empty.")
elif missing_columns:
    msg = f"Validation failed. Missing required columns: {missing_columns}"
    raise KeyError(msg)
else:
    for col in required_columns:
        aggregated_df[col] = pd.to_numeric(aggregated_df[col], errors="coerce")

    invalid_rows = aggregated_df[required_columns].isna().any(axis=1).sum()
    print(f"Validation: {len(aggregated_df)} total rows")
    print(f"Validation: {invalid_rows} rows have invalid/missing likelihood values")

    if invalid_rows:
        print(
            aggregated_df.loc[
                aggregated_df[required_columns].isna().any(axis=1), [*required_columns, "source_file"]
            ].head()
        )

In [ ]:
# Define the function to classify behavior
def get_behaviour(row: pd.Series) -> str:
    behaviour = "No_prob"
    if (row["Tube_prob_likelihood"] >= 0.6) & (row["End_prob_likelihood"] >= 0.6):
        behaviour = "Drinking"
    elif (row["Tube_prob_likelihood"] >= 0.6) ^ (row["End_prob_likelihood"] >= 0.6):
        behaviour = "Prob_out"
    return behaviour


assert aggregated_df is not None, "aggregated_df should be defined at this point"
required_columns = ["Tube_prob_likelihood", "End_prob_likelihood"]
missing_columns = [col for col in required_columns if col not in aggregated_df.columns]

if aggregated_df.empty:
    print("No data loaded; behaviour classification skipped.")
elif missing_columns:
    msg = f"Missing required columns for behaviour classification: {missing_columns}"
    raise KeyError(msg)
else:
    before_count = len(aggregated_df)
    clean_df = aggregated_df.dropna(subset=required_columns).copy()
    dropped_count = before_count - len(clean_df)

    if dropped_count:
        print(f"Dropped {dropped_count} rows with invalid/missing likelihood values before classification.")

    clean_df["Behaviour"] = clean_df.apply(get_behaviour, axis=1)
    aggregated_df = clean_df
    print(aggregated_df["Behaviour"].value_counts(dropna=False))

aggregated_df.head()

In [ ]:
output_path = DOWNLOAD_DIR / "aggregated_journals_with_behaviour.csv"
if not aggregated_df.empty:
    aggregated_df.to_csv(output_path, index=False)
    print(f"Saved aggregated dataset to: {output_path.resolve()}")
else:
    print("Aggregated dataframe is empty; no output written.")